# Cross-encoder reranker ? VSCode + Colab GPU
Connect VSCode to a Colab GPU kernel. Input/output use ZIP files on Google Drive; no upload/download widgets. This notebook calls repository scripts and keeps reporting held out. See docs/RERANKER_BENCHMARK.md for ZIP layout and source preparation. The reranker never reads gold; only the separate evaluator does.

First model: `cross-encoder/ms-marco-MiniLM-L6-v2`. Resolve its revision once below and save a full commit SHA before either dataset runs. No benchmark has been executed in this notebook artifact. Each execution uses new directories and never replaces previous reports.

In [ ]:
from pathlib import Path
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
# If VSCode cannot display Drive authentication, mount in the Colab browser
# on this same runtime first, then return to VSCode.
assert Path('/content/drive/MyDrive').is_dir(), 'Mount Drive before continuing'


In [ ]:
import datetime
import hashlib
import json
import os
import shutil
import subprocess
import sys
import uuid
import zipfile

INPUT_ZIP = Path('/content/drive/MyDrive/retrieval_benchmark/reranker_benchmark_input_v1.zip')
DRIVE_OUTPUT = Path('/content/drive/MyDrive/retrieval_benchmark/reranker_runs')
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
WORK = Path('/content') / ('reranker_' + RUN_ID)
DRIVE_RUN = DRIVE_OUTPUT / RUN_ID
WORK.mkdir(exist_ok=False)
DRIVE_RUN.mkdir(parents=True, exist_ok=False)
assert INPUT_ZIP.is_file(), INPUT_ZIP
# Validate all archive paths before extracting into the fresh workspace.
with zipfile.ZipFile(INPUT_ZIP) as archive:
    for member in archive.infolist():
        target = (WORK / member.filename).resolve()
        assert target.is_relative_to(WORK.resolve()), f'Unsafe archive path: {member.filename}'
        assert (member.external_attr >> 16) & 0o170000 != 0o120000, 'No ZIP symlinks'
    archive.extractall(WORK)
os.chdir(WORK)
required = ['requirements.txt', 'scripts/rerank_retrieval.py', 'scripts/fuse_retrieval_rrf.py',
            'scripts/evaluate_retrieval_benchmark.py']
for dataset in ('qasper', 'scifact'):
    required += [f'configs/dev_{dataset}_seed42.json']
    required += [f'evaluation/reports/dev_{method}_top50_v1/{dataset}_retrieval.json'
                 for method in ('bm25', 'dense')]
for item in required:
    assert (WORK / item).is_file(), f'Missing {item}; ZIP must have project files at root'
print('Workspace:', WORK, 'Drive backup:', DRIVE_RUN)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'Enable Colab GPU; no silent CPU fallback'
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda, 'GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'unittest', 'scripts.test_rerank_retrieval',
                'scripts.test_fuse_retrieval_rrf', '-v'], check=True)


In [ ]:
from huggingface_hub import model_info
from scripts.benchmark_common import file_hash, write_new

CONFIG_PATH = WORK / 'reranker_run_config.json'
# Re-running this cell reuses the saved SHA rather than resolving main again.
if CONFIG_PATH.exists():
    CONFIG = json.loads(CONFIG_PATH.read_text(encoding='utf8'))
else:
    model_id = 'cross-encoder/ms-marco-MiniLM-L6-v2'
    sha = model_info(model_id, revision='main').sha  # metadata only; resolve BEFORE running
    assert len(sha) == 40 and all(c in '0123456789abcdef' for c in sha)
    CONFIG = dict(model_id=model_id, revision=sha, tokenizer_revision=sha,
                  device='cuda', batch_size=16, max_length=512, top_k=10,
                  source_depth=50, bm25_dir='evaluation/reports/dev_bm25_top50_v1',
                  dense_dir='evaluation/reports/dev_dense_top50_v1',
                  report_dir='evaluation/reports/dev_cross_encoder_v1',
                  input_zip_sha256=file_hash(INPUT_ZIP), run_id=RUN_ID,
                  reporting_unlocked=False)
    write_new(CONFIG_PATH, CONFIG)
    write_new(DRIVE_RUN / CONFIG_PATH.name, CONFIG)
assert CONFIG['revision'] == CONFIG['tokenizer_revision']
assert json.loads((DRIVE_RUN / CONFIG_PATH.name).read_text()) == CONFIG
print('Pinned model:', CONFIG['model_id'], CONFIG['revision'])


In [ ]:
REPORT_DIR = WORK / CONFIG['report_dir']

def backup(dataset):
    # New archive per dataset, exclusive creation. Save partial reports on errors too.
    destination = DRIVE_RUN / f'{dataset}_artifacts.zip'
    paths = [CONFIG_PATH, WORK / 'requirements.txt']
    paths += list((WORK / 'scripts').glob('*.py')) + list((WORK / 'app').glob('*.py'))
    paths += [WORK / f'configs/dev_{d}_seed42.json' for d in ('qasper', 'scifact')]
    paths += sorted(REPORT_DIR.glob('*.json'))
    with zipfile.ZipFile(destination, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in paths:
            archive.write(path, path.relative_to(WORK).as_posix())
    write_new(DRIVE_RUN / f'{dataset}_archive_sha256.json',
              {'file': destination.name, 'sha256': file_hash(destination)})
    print('Saved to Drive:', destination)

for dataset in ('qasper', 'scifact'):
    try:
        subprocess.run([sys.executable, '-m', 'scripts.rerank_retrieval',
            '--dataset', dataset, '--bm25-dir', CONFIG['bm25_dir'],
            '--dense-dir', CONFIG['dense_dir'], '--report-dir', CONFIG['report_dir'],
            '--model-id', CONFIG['model_id'], '--revision', CONFIG['revision'],
            '--device', CONFIG['device'], '--batch-size', str(CONFIG['batch_size']),
            '--max-length', str(CONFIG['max_length']), '--top-k', str(CONFIG['top_k'])], check=True)
        subprocess.run([sys.executable, '-m', 'scripts.evaluate_retrieval_benchmark',
            '--dataset', dataset, '--split', 'dev', '--report-dir', CONFIG['report_dir'],
            '--query-manifest', f'configs/dev_{dataset}_seed42.json'], check=True)
    finally:
        backup(dataset)


Reports and pinned configuration are now in the run directory on Drive. Reranking timings exclude retrieval and are not pipeline latency. Evaluation defaults to diagnosis/calibration; do not add `--include-reporting`. Archives can contain diagnosis gold: keep these research artifacts private. To rerun a completed experiment, start a new run directory; output files are immutable. Preserve the input ZIP together with the output archives to reproduce source report hashes.